In [ ]:
!pip install transformers datasets torch pandas


In [ ]:
from google.colab import files
files.upload()   # Upload kaggle.json here



Saving twcs.csv to twcs.csv


In [ ]:
import pandas as pd

# Replace with your uploaded file name
df = pd.read_csv("twcs.csv")

# See first few rows
df.head()


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [ ]:
print("--- Columns After Initial Load ---")
print(df.columns)

--- Columns After Initial Load ---
Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id', 'clean_text'],
      dtype='object')


In [ ]:
import re

# Define the cleaning function
def clean_text(text):
    text = re.sub(r'http\S+|@\S+|#\S+', '', str(text))  # remove URLs, @, #
    text = re.sub(r'[^A-Za-z0-9\s.,?!]', '', text)      # remove special characters
    text = text.lower().strip()
    return text

# Apply the function to the 'text' column to create a new 'clean_text' column
print("Cleaning text...")
df['clean_text'] = df['text'].apply(clean_text)
print("Text cleaning complete.")

# Display the result to verify
print(df[['text', 'clean_text']].head())

Cleaning text...
Text cleaning complete.
                                                text  \
0  @115712 I understand. I would like to assist y...   
1      @sprintcare and how do you propose we do that   
2  @sprintcare I have sent several private messag...   
3  @115712 Please send us a Private Message so th...   
4                                 @sprintcare I did.   

                                          clean_text  
0  i understand. i would like to assist you. we w...  
1                  and how do you propose we do that  
2  i have sent several private messages and no on...  
3  please send us a private message so that we ca...  
4                                             i did.  


In [ ]:
print("\nIdentifying conversation pairs...")

# Condition 1: Find all rows that are customer queries (inbound == True)
is_customer_query = (df['inbound'] == True)

# Condition 2: Find all rows where the *next* row is a company reply (inbound == False)
is_company_reply_next = (df['inbound'].shift(-1) == False)

# Combine the conditions to find the starting index of each valid pair
pair_start_indices = df[is_customer_query & is_company_reply_next].index

# Get the indices for the corresponding replies (which are just the next row)
reply_indices = pair_start_indices + 1

print(f"Found {len(pair_start_indices)} valid conversation pairs.")


Identifying conversation pairs...
Found 1169386 valid conversation pairs.


In [ ]:
print("\nAssembling the final DataFrame...")

# Select the customer queries using their indices
customer_queries = df.loc[pair_start_indices, 'clean_text'].values

# Select the company replies using their indices
company_replies = df.loc[reply_indices, 'clean_text'].values

# Create the final, clean DataFrame
chat_df = pd.DataFrame({
    'query': customer_queries,
    'reply': company_replies
})

print("\n--- Successfully Created Conversation Pairs ---")
print(chat_df.head())


Assembling the final DataFrame...

--- Successfully Created Conversation Pairs ---
                                               query  \
0  i have sent several private messages and no on...   
1                                             i did.   
2                      is the worst customer service   
3  you gonna magically change your connectivity f...   
4          since i signed up with you....since day 1   

                                               reply  
0  please send us a private message so that we ca...  
1  can you please send us a private message, so t...  
2  this is saddening to hear. please shoot us a d...  
3  we understand your concerns and wed like for y...  
4  h there! wed definitely like to work with you ...  


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Libraries for model loading have been imported.")

Libraries for model loading have been imported.


In [ ]:
# The official identifier for the model on the Hugging Face Hub
model_name = "microsoft/DialoGPT-small"

print(f"Selected model: {model_name}")

Selected model: microsoft/DialoGPT-small


In [ ]:
print(f"Downloading tokenizer for '{model_name}'...")

# .from_pretrained() downloads and caches the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer downloaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Tokenizer downloaded successfully.


In [ ]:
print(f"Downloading model weights for '{model_name}'... (This may take a moment)")

# .from_pretrained() downloads and caches the model
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Pre-trained model loaded successfully and is ready to be fine-tuned.")

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Pre-trained model loaded successfully and is ready to be fine-tuned.


In [ ]:
from torch.utils.data import Dataset, DataLoader

print("Imported Dataset and DataLoader from PyTorch.")

Imported Dataset and DataLoader from PyTorch.


In [ ]:
class ChatDataset(Dataset):
    # Part 1: Initialize the dataset with our data and tokenizer
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        self.eos_token = tokenizer.eos_token # The special end-of-string token

    # Part 2: Return the total number of items in the dataset
    def __len__(self):
        return len(self.data)

    # Part 3: Get a single item, format it, and tokenize it
    def __getitem__(self, idx):
        query = self.data.iloc[idx]['query']
        reply = self.data.iloc[idx]['reply']

        # Format the text as "query<|endoftext|>reply<|endoftext|>"
        input_text = query + self.eos_token + reply + self.eos_token

        # Convert the formatted text string into a tensor of token IDs
        input_ids = self.tokenizer.encode(input_text, return_tensors='pt', max_length=128, truncation=True)[0]
        return input_ids

print("Custom ChatDataset class defined successfully.")

Custom ChatDataset class defined successfully.


In [ ]:
# Use a sample of the data to speed up training.
# min() prevents an error if your dataset is smaller than 1000 pairs.
sample_size = min(1000, len(chat_df))
training_sample = chat_df.sample(sample_size)

# Create an object from our class
train_dataset = ChatDataset(training_sample, tokenizer)

print(f"Created a training dataset instance with {len(train_dataset)} conversation pairs.")

Created a training dataset instance with 1000 conversation pairs.


In [ ]:
# The DataLoader will shuffle the data and provide it in batches
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

print("DataLoader is ready. Data is now prepared for model training!")

DataLoader is ready. Data is now prepared for model training!


In [ ]:
import torch

# AdamW is an efficient and popular optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

print("Optimizer (AdamW) has been configured.")

Optimizer (AdamW) has been configured.


In [ ]:
# Puts the model in training mode
model.train()

print("Model set to training mode.")

Model set to training mode.


In [ ]:
# We will run the training for 1 full epoch
num_epochs = 1

print(f"\n--- Starting model fine-tuning for {num_epochs} epoch(s)... ---")

for epoch in range(num_epochs):
    print(f"\n--- Starting Epoch {epoch + 1}/{num_epochs} ---")
    # The inner loop for processing data will go here...



--- Starting model fine-tuning for 1 epoch(s)... ---

--- Starting Epoch 1/1 ---


In [ ]:
# (This code goes inside the epoch loop from Part 6c)

for i, batch in enumerate(train_loader):
    # Step 1 & 2: Forward pass and loss calculation
    outputs = model(batch, labels=batch)
    loss = outputs.loss

    # Step 3: Reset gradients from the previous step
    optimizer.zero_grad()

    # Step 4: Backward pass to calculate gradients
    loss.backward()

    # Step 5: Update the model's weights
    optimizer.step()

    # Optional: Print the loss every 100 steps to monitor progress
    if (i + 1) % 100 == 0:
        print(f"  Step [{i + 1}/{len(train_loader)}], Loss: {loss.item():.4f}")

print("\n--- Fine-Tuning Complete! ---")

  Step [100/1000], Loss: 4.7687
  Step [200/1000], Loss: 4.1078
  Step [300/1000], Loss: 4.1540
  Step [400/1000], Loss: 4.6692
  Step [500/1000], Loss: 3.9627
  Step [600/1000], Loss: 6.6194
  Step [700/1000], Loss: 4.0324
  Step [800/1000], Loss: 3.9350
  Step [900/1000], Loss: 2.6508
  Step [1000/1000], Loss: 3.8245

--- Fine-Tuning Complete! ---


In [ ]:
# Set the model to evaluation mode for inference
model.eval()

print("Model set to evaluation mode. Ready for chatting!")

Model set to evaluation mode. Ready for chatting!


In [ ]:
def chatbot_response(text):
    # Step 1: Encode the user's text into token IDs
    input_ids = tokenizer.encode(text + tokenizer.eos_token, return_tensors='pt')

    # Step 2: Generate a response from the model
    # The model predicts the next token IDs
    output_ids = model.generate(
        input_ids,
        max_length=50,  # Limit the response length to 50 tokens
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3, # Prevent the bot from repeating the same phrases
        early_stopping=True
    )

    # Step 3: Decode the token IDs back into a text string
    # We skip special tokens like <|endoftext|> for a clean output
    response = tokenizer.decode(output_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response

print("Chatbot response function is defined.")

Chatbot response function is defined.


In [ ]:
print("\n--- Your Custom Chatbot is Live! ---")
print("Type 'exit' or 'quit' to end the conversation.")

while True:
    # Prompt the user for their message
    user_input = input("You: ")

    # Check if the user wants to end the chat
    if user_input.lower() in ['exit', 'quit']:
        print("Bot: Goodbye!")
        break

    # Get the bot's response and print it
    bot_reply = chatbot_response(user_input)
    print("Bot:", bot_reply)


--- Your Custom Chatbot is Live! ---
Type 'exit' or 'quit' to end the conversation.
You: hello


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Bot: hi there! i would like to look into this with you. please dm us your account email and well look into it for you.
You: how are u
Bot: sorry for the delay. were here to help!
You: what the hell
Bot: we have a very special team for this.
You: no i dont need
Bot: please dm us your account email and password.
You: no
Bot: hi, please dm us your account email and password. thanks.
You: no
Bot: hi, please dm us your account email and password. thanks.
You: quit
Bot: Goodbye!
